# Lesson 2.3 — Time alignment: frames, timestamps, and episodes

This notebook answers three questions that decide whether a dataset can be trained
on at all:

1. how are observations and actions **paired**?
2. what does a **timestamp** actually mean here?
3. does the declared **frequency** match reality?

The answers to 2 and 3 are not what the metadata claims, and this notebook
demonstrates that rather than asserting it.

Prerequisite: the converted dataset `datasets/lerobot/pickcube/`. The first cell
sets a workspace-local Hugging Face cache, because the default cache location is
not writable in this environment.

## 2.3.0 — Environment setup

In [1]:
import os
from pathlib import Path

from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
REPO_ROOT = PROJECT_ROOT

# LeRobot routes dataset parquet reads through the Hugging Face datasets cache.
# Point it inside the workspace so it stays writable and out of the home directory.
HF_CACHE = REPO_ROOT / ".cache" / "hf"
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_CACHE / "datasets"))

DATASET_ROOT = REPO_ROOT / "datasets" / "lerobot" / "pickcube"
print("repo root    :", REPO_ROOT)
print("dataset root :", DATASET_ROOT, "| exists:", DATASET_ROOT.exists())

repo root    : /home/bowenyuan/Projects/embodied-ai-learning
dataset root : /home/bowenyuan/Projects/embodied-ai-learning/datasets/lerobot/pickcube | exists: True


## 2.3.1 — Frame, timestamp, and episode

- **frame** — one timestep entry: one observation, one action, one timestamp.
- **episode** — a contiguous rollout, from reset to termination or truncation.
- **timestamp** — when frames were *recorded*. Everything about control timing rests
  on this field being real.

LeRobot maps `frame_index` to seconds using the dataset's declared FPS. So the
declared FPS silently defines the time axis of every temporal model built on it.

In [2]:
import numpy as np
import pyarrow.parquet as pq

from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(repo_id="pickcube", root=str(DATASET_ROOT))

print("number of frames  :", len(dataset))
print("number of episodes:", dataset.num_episodes)
print("declared FPS      :", dataset.fps)

sample = dataset[0]
print("\nsample keys:", sorted(sample.keys()))
print("observation.state:", tuple(sample["observation.state"].shape), sample["observation.state"].dtype)
print("action           :", tuple(sample["action"].shape), sample["action"].dtype)
print("timestamp        :", float(sample["timestamp"]), "| frame_index:", int(sample["frame_index"]))

number of frames  : 50
number of episodes: 1
declared FPS      : 50

sample keys: ['action', 'episode_index', 'frame_index', 'index', 'observation.state', 'task', 'task_index', 'timestamp']
observation.state: (42,) torch.float32
action           : (8,) torch.float32
timestamp        : 0.0 | frame_index: 0


## 2.3.2 — What the timestamp actually is

Read the parquet payload directly. If timestamps are real acquisition times, their
intervals will vary slightly. If they are synthetic, the intervals will be exactly
constant.

This distinction is not academic: a constant interval means the numbers were
computed from an index, not measured.

In [3]:
parquet_path = DATASET_ROOT / "data" / "chunk-000" / "file-000.parquet"
table = pq.read_table(parquet_path)
print("parquet schema:")
print(table.schema)

timestamps = table.column("timestamp").to_numpy()
frame_index = table.column("frame_index").to_numpy()
episode_index = table.column("episode_index").to_numpy()

print("\ntimestamp head:", np.round(timestamps[:5], 4))
print("timestamp tail:", np.round(timestamps[-3:], 4))
print("frame_index    :", frame_index[:5], "...", frame_index[-3:])
print("episode_index unique:", np.unique(episode_index))

intervals = np.diff(timestamps)
print("\nintervals (unique values):", np.unique(np.round(intervals, 6)))
print("all intervals identical   :", np.allclose(intervals, intervals[0]))

implied = np.arange(len(timestamps)) * intervals[0]
print("matches np.arange(T) * 0.02:", np.allclose(timestamps, implied, atol=1e-6))

parquet schema:
observation.state: fixed_size_list<element: float>[42]
  child 0, element: float
action: fixed_size_list<element: float>[8]
  child 0, element: float
timestamp: float
frame_index: int64
episode_index: int64
index: int64
task_index: int64
-- schema metadata --
huggingface: '{"info": {"features": {"observation.state": {"feature": {"d' + 458

timestamp head: [0.   0.02 0.04 0.06 0.08]
timestamp tail: [0.94 0.96 0.98]
frame_index    : [0 1 2 3 4] ... [47 48 49]
episode_index unique: [0]

intervals (unique values): [0.02]
all intervals identical   : True
matches np.arange(T) * 0.02: True


## 2.3.3 — The 20 Hz / 50 Hz defect

The timestamps are `np.arange(T) * 0.02`, i.e. synthetic 50 Hz. The simulator's real
control rate is **20 Hz**. Compare the declared value against the environment's own
`control_freq`.

| Quantity | Value | Source |
|---|---:|---|
| declared `fps` in dataset metadata | 50 | inferred from synthetic timestamps |
| real control frequency | 20 | `env.unwrapped.control_freq` |
| real control period | 0.05 s | `env.unwrapped.control_timestep` |

The provenance chain is:

```text
collector stores no timing metadata
    -> timestamps synthesized as np.arange(T) * 0.02
        -> converter infers FPS from those timestamps
            -> info.json declares fps = 50 over 20 Hz data
```

**Consequence:** LeRobot converts `frame_index` to seconds using the declared FPS, so
the time axis is compressed by **2.5x**. A future temporal window `[B, T, D]` would
not cover the number of seconds it appears to cover.

In [4]:
import gymnasium as gym
import mani_skill.envs

env = gym.make("PickCube-v1", obs_mode="state", control_mode="pd_joint_delta_pos", num_envs=1)
unwrapped = env.unwrapped

declared_fps = dataset.fps
real_control_freq = unwrapped.control_freq
real_control_period = unwrapped.control_timestep
sim_freq = unwrapped.sim_freq

print(f"declared FPS in dataset   : {declared_fps}")
print(f"real control_freq         : {real_control_freq} Hz")
print(f"real control_timestep     : {real_control_period} s")
print(f"sim_freq                  : {sim_freq} Hz  (physics steps per second)")
print(f"physics steps per action  : {int(sim_freq * real_control_period)}")
print()
print(f"declared period           : {1 / declared_fps:.4f} s")
print(f"real period               : {real_control_period:.4f} s")
print(f"time axis compression     : {real_control_period / (1 / declared_fps):.1f}x")

seconds_declared = len(dataset) / declared_fps
seconds_real = len(dataset) / real_control_freq
print(f"\n50 frames span {seconds_declared:.2f} s according to metadata, but {seconds_real:.2f} s of real control")
env.close()

2026-09-22 11:36:28,496 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


declared FPS in dataset   : 50
real control_freq         : 20 Hz
real control_timestep     : 0.05 s
sim_freq                  : 100 Hz  (physics steps per second)
physics steps per action  : 5

declared period           : 0.0200 s
real period               : 0.0500 s
time axis compression     : 2.5x

50 frames span 1.00 s according to metadata, but 2.50 s of real control


### Why this cannot be fixed by editing one number

Changing `fps` from 50 to 20 in the metadata would make the declared value true, but
the underlying timestamps would still be synthetic. The durable fix is at the source:

- the **collector** must record the control frequency it actually used, and either
  measure or explicitly label its timestamps;
- the **converter** must declare the frequency from that recorded value instead of
  inferring it from timestamps it is about to fabricate.

Until then, treat the dataset as **50 frames of 20 Hz control**, and treat the
declared 50 FPS as wrong. This is recorded as an open issue in `notes/progress.md`.

## 2.3.4 — The training pair is `(o_t, a_t)`, not `(o_t, a_{t+1})`

The pairing convention is a contract, not a convention. If actions are shifted by
one frame, every target is wrong while all shapes stay identical — the model trains
happily and learns to predict the next action.

Verify the rule on the stored data: the action at index `t` is the action that was
applied to the observation at index `t`.

In [5]:
# The stored arrays are aligned by construction, so demonstrate the contrast
# between the correct pairing and the off-by-one pairing.

actions = table.column("action").to_pylist()
states = table.column("observation.state").to_pylist()

T = len(actions)
print("T actions :", T)
print("T states  :", len(states))

correct_pairs = [(t, t) for t in range(T)]
shifted_pairs = [(t, t + 1) for t in range(T - 1)]

print("\ncorrect pairing  : (o_t, a_t)     ->", correct_pairs[:3], "...")
print("off-by-one pairing: (o_t, a_t+1)   ->", shifted_pairs[:3], "...")
print("\nboth produce the same tensor shapes;")
print("only the first matches the environment transition o_t -a_t-> o_{t+1}.")

T actions : 50
T states  : 50

correct pairing  : (o_t, a_t)     -> [(0, 0), (1, 1), (2, 2)] ...
off-by-one pairing: (o_t, a_t+1)   -> [(0, 1), (1, 2), (2, 3)] ...

both produce the same tensor shapes;
only the first matches the environment transition o_t -a_t-> o_{t+1}.


### The `T` versus `T+1` question

An episode of T actions involves **T+1** states:

```text
o_0 --a_0--> o_1 --a_1--> o_2 ... o_{T-1} --a_{T-1}--> o_T
```

The stored observation array holds `o_0 .. o_{T-1}` — the states the policy *saw*.
The terminal observation `o_T` (and `next_observations` in the raw HDF5) is a
separate field. Confusing the two produces a dataset that looks complete and is
silently missing its last transition.

In [6]:
import h5py

h5_path = REPO_ROOT / "datasets" / "pickcube" / "random_episode_standard.h5"
with h5py.File(h5_path, "r") as handle:
    print("standardized HDF5 keys and shapes:")
    for key in handle:
        if key == "metadata":
            print(f"  {key:<18} (group) attrs={dict(handle[key].attrs)}")
        else:
            print(f"  {key:<18} shape={handle[key].shape} dtype={handle[key].dtype}")

print("\nNote timestamps were stored as a separate array:")
with h5py.File(h5_path, "r") as handle:
    stamps = handle["timestamps"][:]
print("  first 5:", stamps[:5])
print("  step    :", np.unique(np.round(np.diff(stamps), 6)))

standardized HDF5 keys and shapes:
  actions            shape=(50, 8) dtype=float32
  metadata           (group) attrs={'control_mode': 'pd_joint_delta_pos', 'robot': 'Panda', 'source': 'ManiSkill', 'task': 'PickCube-v1'}
  observations       shape=(50, 42) dtype=float32
  rewards            shape=(50, 1) dtype=float32
  timestamps         shape=(50,) dtype=float64

Note timestamps were stored as a separate array:
  first 5: [0.   0.02 0.04 0.06 0.08]
  step    : [0.02]


## Takeaways

1. The dataset's timestamps are synthetic (`np.arange(T) * 0.02`), not measured — the
   intervals are exactly constant, which is the giveaway.
2. The declared 50 FPS contradicts the environment's real **20 Hz** control rate,
   compressing the time axis by 2.5x. Do not silently edit `fps`; fix the collector
   and the converter.
3. Training pairs are `(o_t, a_t)`. The off-by-one pairing has identical shapes and
   is wrong.
4. An episode of T actions has T+1 states. The stored observation array has T.

Next: `2.4_observation_schema.ipynb` for the observation/action schema, then
`2.5_generate_lerobot_dataset.ipynb` for the conversion.